# Day05 下午个人项目：电商用户多维分析

**姓名：** 刘佳琦  
**专题方向：** B（投诉与服务体验）

本 Notebook 已完成数据验收、基础指标、单维分析、双维交叉分析、报表导出、结论、限制与建议。


## 一、实验目标与提交要求

你需要独立完成：

1. 读取并验收第4天清洗后的数据；
2. 计算公共基础指标；
3. 选择一个专题完成单维分析；
4. 完成至少一个双维度交叉分析；
5. 输出三个标准CSV报表；
6. 撰写至少3条结论、1条限制和1项建议；
7. 将Notebook和输出文件提交到个人GitHub仓库。

### 必须遵守的分析边界

- 一行数据代表一名用户，不是一笔订单；
- `CustomerID`是标识符，不适合求平均值；
- `CashbackAmount`是返现金额，不是消费金额或销售额；
- 当前数据没有订单金额和订单日期，不能计算GMV、客单价或时间趋势；
- 分组差异只能说明关联，不能直接证明因果关系；
- 所有比例表必须同时包含样本量。

## 二、专题方向

| 专题 | 推荐字段 | 参考业务问题 |
|---|---|---|
| A 用户生命周期 | `TenureGroup` | 不同生命周期用户的流失和订单行为有何差异？ |
| B 投诉与服务体验 | `Complain`、`SatisfactionScore` | 投诉、满意度与流失存在怎样的关联？ |
| C 品类与订单行为 | `PreferedOrderCat` | 不同偏好品类用户的规模和订单行为有何差异？ |
| D 支付与优惠行为 | `PreferredPaymentMode` | 支付偏好与优惠行为是否存在分组差异？ |
| E 城市与设备行为 | `CityTier`、`PreferredLoginDevice` | 城市、设备与用户活跃或流失有何关联？ |

请选择一个专题作为单维分析主线。双维分析可以在此基础上增加另一个业务维度。

## 任务0：个人配置与运行环境

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display, Markdown
except ImportError:
    def display(obj):
        print(obj)

    def Markdown(text):
        return text


# =========================
# 个人信息与专题
# =========================
STUDENT_NAME = "刘佳琦"
TOPIC = "B"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        data_path = (
            candidate
            / "output"
            / "day04_project"
            / "ecommerce_customer_cleaned.csv"
        )

        if data_path.exists():
            return candidate

    raise FileNotFoundError(
        "未找到清洗后数据，请检查："
        "output/day04_project/ecommerce_customer_cleaned.csv"
    )


ROOT = find_workspace_root()
DATA_PATH = (
    ROOT
    / "output"
    / "day04_project"
    / "ecommerce_customer_cleaned.csv"
)
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)


姓名： 刘佳琦
专题： B
输入数据： /Users/xuyifan/Desktop/muc-commerce-3-24012460/output/day04_project/ecommerce_customer_cleaned.csv
输出目录： /Users/xuyifan/Desktop/muc-commerce-3-24012460/output/day05_analysis


In [5]:
# 检查点0：个人信息与专题配置

assert STUDENT_NAME != "请填写姓名", "请填写STUDENT_NAME"
assert STUDENT_NAME.strip(), "姓名不能为空"

TOPIC = TOPIC.strip().upper()
assert TOPIC in {"A", "B", "C", "D", "E"}, \
    "TOPIC只能填写A、B、C、D或E"

expected_output_dir = ROOT / "output" / "day05_analysis"
assert OUTPUT_DIR == expected_output_dir, \
    "输出目录应为output/day05_analysis"

print("检查点0通过")
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)

检查点0通过
姓名： 刘佳琦
专题： B


### 检查点0完成标志

- [x] 已填写姓名；
- [x] `TOPIC`只填写A、B、C、D或E；
- [x] 输出目录为`output/day05_analysis`；
- [x] Notebook文件名保持为`day05_pm_student_project.ipynb`。


## 任务1：读取并验收数据（必做）

In [3]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

数据形状： (5630, 22)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,成长期(4-6个月),1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,稳定期(7-12个月),1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,稳定期(7-12个月),1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,新用户(≤3个月),1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,新用户(≤3个月),1



字段类型：


,数据类型
CustomerID,int64
Churn,int64
Tenure,float64
PreferredLoginDevice,object
CityTier,int64
WarehouseToHome,float64
PreferredPaymentMode,object
Gender,object
HourSpendOnApp,float64
NumberOfDeviceRegistered,int64


In [4]:
# 定义需要验收的核心字段
core_cols = [
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
]


# 完成数据验收表
validation = pd.DataFrame({
    "验收项目": [
        "数据行数",
        "数据列数",
        "CustomerID重复数",
        "核心字段缺失数",
        "Churn取值",
    ],
    "验收结果": [
        df.shape[0],
        df.shape[1],
        int(df["CustomerID"].duplicated().sum()),
        int(df[core_cols].isna().sum().sum()),
        sorted(df["Churn"].dropna().unique().tolist()),
    ],
})


# 展示验收结果
display(validation)


,验收项目,验收结果
0,数据行数,5630
1,数据列数,22
2,CustomerID重复数,0
3,核心字段缺失数,0
4,Churn取值,"[0, 1]"


In [5]:
# 检查点1：数据结构与核心质量

assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应保持唯一"
assert set(df["Churn"].unique()) == {0, 1}, \
    "Churn应只包含0和1"

required_core_cols = {
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
}

assert required_core_cols.issubset(core_cols), \
    f"core_cols缺少字段：{required_core_cols - set(core_cols)}"
assert df[core_cols].notna().all().all(), \
    "核心分析字段仍存在缺失值"
assert validation is not None, "请完成validation验收表"

print("检查点1通过")

检查点1通过


### 数据粒度说明

一行数据代表一名独立的电商用户，而不是一笔订单。

`CustomerID`只是用于唯一识别用户的标识符，其数值大小不具有连续业务含义，因此不能将其作为普通连续变量求平均。


## 任务2：公共基础指标（必做）

请构建`overall_metrics`，至少包含以下10项指标：

1. 用户数；
2. 流失人数；
3. 总体流失率；
4. 平均订单数；
5. 订单数中位数；
6. 平均优惠券使用次数；
7. 平均返现；
8. 平均App使用时长；
9. 平均满意度；
10. 平均距上次下单天数。

输出建议使用“指标、数值”两列的DataFrame。

In [7]:
# 计算公共基础指标
overall_churn_rate = float(df["Churn"].mean())

overall_metrics = pd.DataFrame({
    "指标": [
        "用户数",
        "流失人数",
        "总体流失率",
        "平均订单数",
        "订单数中位数",
        "平均优惠券使用次数",
        "平均返现",
        "平均App使用时长",
        "平均满意度",
        "平均距上次下单天数",
    ],
    "数值": [
        int(df["CustomerID"].nunique()),
        int(df["Churn"].sum()),
        overall_churn_rate,
        float(df["OrderCount"].mean()),
        float(df["OrderCount"].median()),
        float(df["CouponUsed"].mean()),
        float(df["CashbackAmount"].mean()),
        float(df["HourSpendOnApp"].mean()),
        float(df["SatisfactionScore"].mean()),
        float(df["DaySinceLastOrder"].mean()),
    ],
})


# 展示结果
display(overall_metrics)


,指标,数值
0,用户数,"5,630.00"
1,流失人数,948.00
2,总体流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券使用次数,1.72
6,平均返现,177.22
7,平均App使用时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


In [8]:
# 检查点2：公共基础指标

assert isinstance(overall_metrics, pd.DataFrame), \
    "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, \
    "公共基础指标至少包含10项"

overall_churn_rate = float(df["Churn"].mean())

assert overall_churn_rate is not None, \
    "请填写overall_churn_rate"
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, \
    "总体流失率计算不正确"

print("检查点2通过")


检查点2通过


### 公共指标初步观察

当前样本共有5,630名用户，其中流失用户占比约为16.84%；该结果仅描述样本总体情况，不对流失原因作推断。


## 任务3：单维专题分析（必做）

根据所选专题确定一个主分组字段，并使用`groupby + agg`完成命名聚合。

最低要求：

- 必须包含“用户数”；
- 至少再包含3项业务指标；
- 如果包含流失率或占比，必须保留0～1原始小数用于导出；
- 按业务意义排序；
- 分组字段在`reset_index()`后应保留为普通列。

In [9]:
topic_fields = {
    "A": {"TenureGroup"},
    "B": {"Complain", "SatisfactionScore"},
    "C": {"PreferedOrderCat"},
    "D": {"PreferredPaymentMode"},
    "E": {"CityTier", "PreferredLoginDevice"},
}

print("当前专题：", TOPIC)
print("可选主分组字段：", topic_fields[TOPIC])


# 选择主分组字段
segment_field = "Complain"


# 使用groupby + agg完成命名聚合
segment_analysis = (
    df.groupby(segment_field, dropna=False)
    .agg(
        用户数=("CustomerID", "nunique"),
        流失人数=("Churn", "sum"),
        流失率=("Churn", "mean"),
        平均满意度=("SatisfactionScore", "mean"),
        平均订单数=("OrderCount", "mean"),
        平均返现=("CashbackAmount", "mean"),
    )
    .reset_index()
)


# 按投诉状态排序并展示
segment_analysis = segment_analysis.sort_values(
    by=segment_field,
    ascending=True,
).reset_index(drop=True)

display(segment_analysis)


当前专题： B
可选主分组字段： {'SatisfactionScore', 'Complain'}


,Complain,用户数,流失人数,流失率,平均满意度,平均订单数,平均返现
0,0,4026,440,0.11,3.09,3.00,177.21
1,1,1604,508,0.32,3.00,2.86,177.26


In [10]:
# 检查点3：单维专题分析

assert segment_field in df.columns, \
    "segment_field不是有效字段"
assert segment_field in topic_fields[TOPIC], \
    f"专题{TOPIC}建议使用字段：{topic_fields[TOPIC]}"
assert isinstance(segment_analysis, pd.DataFrame), \
    "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, \
    "专题分析表必须包含用户数"
assert len(segment_analysis) >= 2, \
    "专题分析至少应包含两个分组"
assert segment_analysis["用户数"].sum() == len(df), \
    "各分组用户数之和应等于总用户数"

print("检查点3通过")

检查点3通过


### 单维专题分析记录

**本专题要回答的业务问题：**

投诉用户与未投诉用户在用户规模、流失率、满意度、订单行为和返现金额方面是否存在值得关注的分组差异？

**数据现象与可能解释：**

下方代码会直接读取`segment_analysis`中的真实结果，自动生成包含群体、用户数、指标和具体数值的描述，避免手工抄写造成错误。相关差异只能说明投诉状态与流失等指标存在关联，可能反映服务体验问题，但仍需结合投诉类型、处理时长和用户历史行为进一步验证。


In [11]:
# 根据单维分析结果自动生成带具体数值的文字结论
segment_sorted = segment_analysis.sort_values(
    ["流失率", "用户数"],
    ascending=[False, False],
).reset_index(drop=True)

segment_high = segment_sorted.iloc[0]
segment_low = segment_sorted.iloc[-1]

segment_narrative = (
    f"数据现象：{segment_field}={segment_high[segment_field]}的群体共有"
    f"{int(segment_high['用户数'])}名用户，流失率为{segment_high['流失率']:.2%}，"
    f"平均满意度为{segment_high['平均满意度']:.2f}，平均订单数为"
    f"{segment_high['平均订单数']:.2f}；相比之下，"
    f"{segment_field}={segment_low[segment_field]}的群体共有"
    f"{int(segment_low['用户数'])}名用户，流失率为{segment_low['流失率']:.2%}。\n\n"
    "可能解释：投诉状态与较高或较低流失率相关，值得关注，但不能据此认定"
    "投诉直接导致流失，还需结合投诉内容、处理结果和用户既往行为验证。"
)

display(Markdown(segment_narrative))


数据现象：Complain=1.0的群体共有1604名用户，流失率为31.67%，平均满意度为3.00，平均订单数为2.86；相比之下，Complain=0.0的群体共有4026名用户，流失率为10.93%。

可能解释：投诉状态与较高或较低流失率相关，值得关注，但不能据此认定投诉直接导致流失，还需结合投诉内容、处理结果和用户既往行为验证。

## 任务4：双维度交叉分析（必做）

从以下维度中选择两个不同字段：

- `TenureGroup`
- `Complain`
- `PreferedOrderCat`
- `CityTier`
- `PreferredLoginDevice`
- `PreferredPaymentMode`

最低要求：

- 输出两个分组维度；
- 输出用户数、流失人数、流失率和至少1项行为指标；
- 将用户数少于30的组合标记为“小样本”，其余标记为“可观察”；
- 不得只展示流失率而省略用户数。

In [12]:
allowed_cross_fields = {
    "TenureGroup",
    "Complain",
    "PreferedOrderCat",
    "CityTier",
    "PreferredLoginDevice",
    "PreferredPaymentMode",
}


# 选择两个不同维度
dim_1 = "Complain"
dim_2 = "TenureGroup"


# 使用groupby + agg完成双维分析
cross_analysis = (
    df.groupby([dim_1, dim_2], dropna=False)
    .agg(
        用户数=("CustomerID", "nunique"),
        流失人数=("Churn", "sum"),
        流失率=("Churn", "mean"),
        平均订单数=("OrderCount", "mean"),
        平均满意度=("SatisfactionScore", "mean"),
    )
    .reset_index()
)


# 新增“样本提示”列
cross_analysis["样本提示"] = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察",
)


# 按流失率降序、用户数降序排序并展示
cross_analysis = cross_analysis.sort_values(
    ["流失率", "用户数"],
    ascending=[False, False],
).reset_index(drop=True)

display(cross_analysis)


,Complain,TenureGroup,用户数,流失人数,流失率,平均订单数,平均满意度,样本提示
0,1,新用户(≤3个月),522,345,0.66,2.55,3.07,可观察
1,0,新用户(≤3个月),1038,308,0.30,2.20,3.23,可观察
2,1,成长期(4-6个月),137,30,0.22,3.03,2.75,可观察
3,1,稳定期(7-12个月),406,81,0.20,2.67,3.02,可观察
4,1,忠诚期(13-24个月),414,52,0.13,3.35,2.91,可观察
5,0,稳定期(7-12个月),1178,75,0.06,2.78,2.98,可观察
6,0,忠诚期(13-24个月),1053,43,0.04,3.85,3.16,可观察
7,0,成长期(4-6个月),453,14,0.03,2.95,2.99,可观察
8,0,长期用户(>24个月),304,0,0.00,3.75,3.00,可观察
9,1,长期用户(>24个月),125,0,0.00,3.06,3.19,可观察


In [13]:
# 检查点4：双维度交叉分析

assert dim_1 in allowed_cross_fields and dim_2 in allowed_cross_fields, \
    "两个分析维度必须来自允许字段"
assert dim_1 != dim_2, "两个分析维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), \
    "cross_analysis应为DataFrame"

required_cross_cols = {
    dim_1,
    dim_2,
    "用户数",
    "流失率",
    "样本提示",
}

assert required_cross_cols.issubset(cross_analysis.columns), \
    f"双维分析表缺少字段：{required_cross_cols - set(cross_analysis.columns)}"
assert cross_analysis["用户数"].sum() == len(df), \
    "双维组合用户数之和应等于总用户数"
assert set(cross_analysis["样本提示"]).issubset(
    {"小样本", "可观察"}
), "样本提示只能是“小样本”或“可观察”"

expected_sample_hint = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察",
)
assert np.array_equal(
    cross_analysis["样本提示"].to_numpy(),
    expected_sample_hint,
), "样本提示与用户数阈值不一致"

print("检查点4通过")

检查点4通过


### 双维分析记录

**分析思路：**

将投诉状态与用户生命周期交叉，识别不同生命周期中投诉用户和未投诉用户的流失差异。下方代码会自动提取流失率最高与最低的组合，输出用户数、流失率、比较对象和小样本判断。

**为什么不能直接写成因果结论：**

当前分析属于观察性分组比较，没有随机实验或控制其他混杂因素；生命周期、历史订单行为和服务接触等因素都可能同时影响投诉与流失，因此只能表述为关联。


In [14]:
# 根据双维分析结果自动生成带具体数值的文字结论
cross_sorted = cross_analysis.sort_values(
    ["流失率", "用户数"],
    ascending=[False, False],
).reset_index(drop=True)

cross_high = cross_sorted.iloc[0]
cross_low = cross_sorted.iloc[-1]

small_sample_rows = cross_analysis.loc[
    cross_analysis["样本提示"].eq("小样本")
]

cross_narrative = (
    f"最值得关注的组合是{dim_1}={cross_high[dim_1]}、"
    f"{dim_2}={cross_high[dim_2]}：共有{int(cross_high['用户数'])}名用户，"
    f"流失率为{cross_high['流失率']:.2%}，平均订单数为"
    f"{cross_high['平均订单数']:.2f}。作为比较，"
    f"{dim_1}={cross_low[dim_1]}、{dim_2}={cross_low[dim_2]}共有"
    f"{int(cross_low['用户数'])}名用户，流失率为{cross_low['流失率']:.2%}。\n\n"
    f"小样本风险：共有{len(small_sample_rows)}个组合的用户数少于30，"
    "这些组合的比例容易受少量用户变化影响，应谨慎解释。"
)

display(Markdown(cross_narrative))


最值得关注的组合是Complain=1、TenureGroup=新用户(≤3个月)：共有522名用户，流失率为66.09%，平均订单数为2.55。作为比较，Complain=1、TenureGroup=长期用户(>24个月)共有125名用户，流失率为0.00%。

小样本风险：共有0个组合的用户数少于30，这些组合的比例容易受少量用户变化影响，应谨慎解释。

## 任务5：输出统计报表（必做）

In [15]:
# 输出三个标准CSV文件

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print("已输出：", path.relative_to(ROOT))

已输出： output/day05_analysis/overall_metrics.csv
已输出： output/day05_analysis/segment_analysis.csv
已输出： output/day05_analysis/cross_analysis.csv


In [16]:
# 检查点5：输出文件与回读验证

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename

    assert path.exists(), f"缺少输出文件：{filename}"

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape, \
        f"{filename}回读后的形状与原表不一致"
    assert not any(
        str(col).startswith("Unnamed")
        for col in reloaded.columns
    ), f"{filename}包含多余索引列，请使用index=False导出"

    print(f"通过：{filename}，形状为{reloaded.shape}")

print("检查点5通过")

通过：overall_metrics.csv，形状为(10, 2)
通过：segment_analysis.csv，形状为(2, 7)
通过：cross_analysis.csv，形状为(10, 8)
检查点5通过


## 任务6：结论、限制与建议（必做）

### 结论、限制与建议

下方代码会依据本次实际运行得到的`overall_metrics`、`segment_analysis`和`cross_analysis`自动生成三条定量结论，并给出分析限制和运营建议。这样可以保证文字中的样本量和比例与导出的CSV报表保持一致。


In [17]:
# 自动生成三条结论、限制与建议
segment_ranked = segment_analysis.sort_values(
    ["流失率", "用户数"],
    ascending=[False, False],
).reset_index(drop=True)
cross_ranked = cross_analysis.sort_values(
    ["流失率", "用户数"],
    ascending=[False, False],
).reset_index(drop=True)

seg_top = segment_ranked.iloc[0]
seg_bottom = segment_ranked.iloc[-1]
cross_top = cross_ranked.iloc[0]

conclusions = f"""
### 结论1
当前样本共有{len(df):,}名用户，流失人数为{int(df['Churn'].sum()):,}人，
总体流失率为{overall_churn_rate:.2%}。对应证据表：`overall_metrics.csv`。

### 结论2
在{segment_field}={seg_top[segment_field]}的用户中，用户数为
{int(seg_top['用户数']):,}人，流失率为{seg_top['流失率']:.2%}；
{segment_field}={seg_bottom[segment_field]}的用户数为
{int(seg_bottom['用户数']):,}人，流失率为{seg_bottom['流失率']:.2%}。
两组存在分组差异，但该差异只表示关联。对应证据表：`segment_analysis.csv`。

### 结论3
在{dim_1}={cross_top[dim_1]}且{dim_2}={cross_top[dim_2]}的组合中，
用户数为{int(cross_top['用户数']):,}人，流失率为{cross_top['流失率']:.2%}，
平均订单数为{cross_top['平均订单数']:.2f}。对应证据表：`cross_analysis.csv`。

### 分析限制
当前数据一行代表一名用户，且没有订单金额、订单明细和订单日期，因此不能计算GMV、
客单价或时间趋势；分组比较也未控制其他潜在混杂因素，不能直接证明投诉或生命周期
是用户流失的原因。用户数少于30的交叉组合还存在比例不稳定风险。

### 运营建议与验证方式
建议优先对高流失率且样本量达到“可观察”标准的投诉相关群体开展服务回访和问题分类，
重点记录投诉类型、首次响应时间、解决时长及回访结果。后续可通过小规模随机对照实验，
比较干预组与对照组的留存率、复购率和投诉复发率，以验证服务改进是否真正降低流失。
"""

display(Markdown(conclusions))



### 结论1
当前样本共有5,630名用户，流失人数为948人，
总体流失率为16.84%。对应证据表：`overall_metrics.csv`。

### 结论2
在Complain=1.0的用户中，用户数为
1,604人，流失率为31.67%；
Complain=0.0的用户数为
4,026人，流失率为10.93%。
两组存在分组差异，但该差异只表示关联。对应证据表：`segment_analysis.csv`。

### 结论3
在Complain=1且TenureGroup=新用户(≤3个月)的组合中，
用户数为522人，流失率为66.09%，
平均订单数为2.55。对应证据表：`cross_analysis.csv`。

### 分析限制
当前数据一行代表一名用户，且没有订单金额、订单明细和订单日期，因此不能计算GMV、
客单价或时间趋势；分组比较也未控制其他潜在混杂因素，不能直接证明投诉或生命周期
是用户流失的原因。用户数少于30的交叉组合还存在比例不稳定风险。

### 运营建议与验证方式
建议优先对高流失率且样本量达到“可观察”标准的投诉相关群体开展服务回访和问题分类，
重点记录投诉类型、首次响应时间、解决时长及回访结果。后续可通过小规模随机对照实验，
比较干预组与对照组的留存率、复购率和投诉复发率，以验证服务改进是否真正降低流失。


## 拓展任务（选做）

In [18]:
# 拓展分析：检查投诉群体与未投诉群体的核心指标差值
complain_compare = (
    segment_analysis
    .set_index(segment_field)[
        ["流失率", "平均满意度", "平均订单数", "平均返现"]
    ]
)

if set(complain_compare.index).issuperset({0, 1}):
    complain_gap = (
        complain_compare.loc[1] - complain_compare.loc[0]
    ).to_frame("投诉用户减未投诉用户")
    display(complain_gap)
else:
    print("投诉状态不是标准的0/1取值，请根据实际取值检查分组结果。")


,投诉用户减未投诉用户
流失率,0.21
平均满意度,-0.10
平均订单数,-0.14
平均返现,0.06


## 最终检查：GitHub提交前验收

In [19]:
required_files = [
    ROOT / "notebooks" / "day05_pm_student_project.ipynb",
    OUTPUT_DIR / "overall_metrics.csv",
    OUTPUT_DIR / "segment_analysis.csv",
    OUTPUT_DIR / "cross_analysis.csv",
]

missing_files = [
    str(path.relative_to(ROOT))
    for path in required_files
    if not path.exists()
]

assert not missing_files, \
    f"提交内容不完整，缺少文件：{missing_files}"

for csv_path in required_files[1:]:
    check_df = pd.read_csv(csv_path)
    assert not any(
        str(col).startswith("Unnamed")
        for col in check_df.columns
    ), f"{csv_path.name}仍包含多余索引列"

print("本地提交文件检查通过")
print("请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。")

本地提交文件检查通过
请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。


### GitHub提交清单

- [x] 已填写姓名和专题；
- [ ] Notebook已在包含清洗后数据的课程仓库中重启内核并从头运行成功；
- [ ] 所有检查点均已通过；
- [ ] `output/day05_analysis/`中包含三个CSV；
- [ ] CSV中没有`Unnamed`索引列；
- [x] 已准备至少3条结论、1条限制和1项建议；
- [x] 没有把返现写成消费额；
- [x] 没有把相关关系写成确定因果关系；
- [ ] 已提交并推送到个人GitHub仓库。

### 最终反思

1. 本次分析最重要的数据发现，是投诉状态及其与用户生命周期的交叉组合在流失率上可能存在明显分组差异。
2. 数据结构与核心质量检查点最能帮助发现错误，因为它会同时检查数据形状、主键唯一性、标签取值和核心字段缺失。
3. “投诉用户流失率更高”最容易被误解为因果关系；当前结果只能说明关联，不能证明投诉直接导致流失。
4. 最希望增加投诉类型、投诉时间、解决时长和解决结果字段，以判断具体服务问题及处理效率。
5. 第6天准备优先将`cross_analysis.csv`转化为分组柱状图或热力图，因为它能同时呈现投诉状态、生命周期、样本量和流失率。
